![](https://raw.githubusercontent.com/Databricks-BR/workshop_agents/refs/heads/main/demo-main/img/header_workshop.png)

### Seu database (schema no Unity Catalog)

Cada participante usa um **database** próprio. No Unity Catalog, isso é o **nome do schema** dentro do catálogo `dbacademy`.

**Regra:** **primeira letra do seu nome** + **seu sobrenome**, em **minúsculas**, sem espaços.

| Exemplo | Database |
|---------|----------|
| Aluno: **Carlos Bettanim** | `cbettanim` |

**Anote o seu identificador** — você vai usá-lo em **todos** os notebooks e referências.

Nos códigos SQL/Python deste notebook, substitua **`<seu_database>`** pelo seu valor (ex.: `cbettanim`). O objeto fica no formato `dbacademy.<seu_database>` → `dbacademy.cbettanim`.

> **Dica:** no Databricks, use *Edit → Find and Replace* no notebook para trocar `<seu_database>` de uma vez.

**Importante:** enquanto `<seu_database>` não for substituído por um nome válido (ex.: `cbettanim`), as células SQL/Python **não** devem ser executadas — o catálogo `dbacademy.<seu_database>` não é um identificador válido no Unity Catalog.


   
# Gerar Dados

   

 Item | Descrição |
 --- | --- |
 **Objetivo** | Gerar Tabelas de Vendas e Logística |
 **Databricks Run Time** | DBR 16.4 LTS |
 **Linguagem** | Python, Pyspark e SQL |

![](https://raw.githubusercontent.com/Databricks-BR/workshop_agents/refs/heads/main/demo-main/img/img/01_diagram_2.png)

   
## Gerar Tabelas

In [0]:
import pandas as pd
import random
from datetime import date, timedelta

In [0]:
# --- Configuração ---
NUM_PRODUCTS = 500
NUM_STORES = 100
NUM_INVENTORY_RECORDS = 10000
NUM_SALES_RECORDS = 10000

In [0]:
# --- Geração de Dados de Produtos ---
def generate_product_data(num_products):
    """Gera dados de exemplo de produtos."""
    
    product_types = [
        "Capa para Celular", "Fones de Ouvido Sem Fio", "Teclado Mecânico", "Mouse Gamer",
        "Monitor LED", "Cabo USB-C", "Carregador Rápido", "Mochila Urbana", "Garrafa Térmica de Aço",
        "Cuia de Chimarrão", "Bomba de Alpaca", "Camiseta de Algodão", "Calça Jeans",
        "Tênis Esportivo", "Luminária de Mesa", "Cadeira Ergonômica", "Garrafa Térmica",
        "Caixa de Som Bluetooth", "Smartwatch", "Tablet 10 Polegadas", "Webcam HD"
    ]
    
    product_descriptors = [
        "Resistente", "Bluetooth 5.0", "RGB", "Ergonômico", "4K Ultra HD",
        "Reforçado", "Inteligente", "Impermeável", "1 Litro", "Premium",
        "Bico de Papagaio", "Estampada", "Slim Fit", "Para Corrida", "Luz Quente",
        "Suporte Lombar", "500ml", "Som Envolvente", "GPS Integrado",
        "Android 12", "com Microfone"
    ]
    
    data = []
    product_ids = []
    for i in range(num_products):
        # Converte product_id para string para evitar erros de conversão Arrow no Spark
        product_id = str(random.randint(10**18, 10**19 - 1))
        product_ids.append(product_id)
        
        product_name = f"{random.choice(product_types)} {random.choice(product_descriptors)}"
        
        data.append({
            "product_id": product_id,
            "supplier": f"FORNECEDOR {random.randint(1, 20)}",
            "product": product_name,
            "upc": str(random.randint(10**12, 10**13 - 1)).zfill(13)
        })
    return pd.DataFrame(data), product_ids

In [0]:
# --- Geração de Dados de Lojas ---
def generate_store_data(num_stores):
    """Gera dados de exemplo de lojas com localizações no Brasil."""
    data = []
    store_ids = []
    for i in range(num_stores):
        # Converte store_id para string para evitar erros de conversão Arrow no Spark
        store_id = str(random.randint(10**18, 10**19 - 1))
        store_ids.append(store_id)
        
        # Faixas de latitude e longitude do Brasil
        lat = random.uniform(-33.0, 5.0)
        lon = random.uniform(-74.0, -35.0)
        
        data.append({
            "store_id": store_id,
            "store": f"LOJA {random.randint(10000, 99999)}",
            "store_type": f"TIPO LOJA {random.randint(1, 30)}",
            "store_zip": f"{random.randint(10000000, 99999999):08d}", # Formato CEP no Brasil
            "store_lat_long": f"{lat:.2f},{lon:.2f}",
            "retailer": f"VAREJISTA {random.randint(1, 5)}"
        })
    return pd.DataFrame(data), store_ids

In [0]:
# --- Geração da Tabela Fato de Inventário ---
def generate_inventory_data(num_records, store_ids, product_ids):
    """Gera dados de exemplo de inventário, vinculando lojas e produtos."""
    
    start_date = date(2021, 1, 1)
    end_date = date(2023, 12, 31)
    date_range = (end_date - start_date).days
    
    data = []
    for _ in range(num_records):
        random_days = random.randrange(date_range)
        
        data.append({
            # Converte inventory_id para string para evitar erros de conversão Arrow no Spark
            "inventory_id": str(random.randint(10**18, 10**19 - 1)),
            "store_id": random.choice(store_ids),
            "product_id": random.choice(product_ids),
            "date_key": start_date + timedelta(days=random_days),
            "on_hand_quantity": random.randint(0, 100)
        })
    return pd.DataFrame(data)

In [0]:
# --- Geração da Tabela Fato de Vendas ---
def generate_sales_data(num_records, store_ids, product_ids):
    """Gera dados de exemplo de vendas, vinculando lojas e produtos."""
    
    start_date = date(2021, 1, 1)
    end_date = date(2023, 12, 31)
    date_range = (end_date - start_date).days
    
    data = []
    for _ in range(num_records):
        random_days = random.randrange(date_range)
        quantity = random.randint(1, 10)
        price = random.uniform(5.0, 500.0)
        
        data.append({
            # Converte sales_id para string para evitar erros de conversão Arrow no Spark
            "sales_id": str(random.randint(10**10, 10**11 - 1)),
            "store_id": random.choice(store_ids),
            "product_id": random.choice(product_ids),
            "date_key": start_date + timedelta(days=random_days),
            "sales_quantity": quantity,
            "sales_amount": quantity * price
        })
    return pd.DataFrame(data)

In [0]:
# --- Bloco de execução principal ---
if __name__ == "__main__":
    print("Gerando tabelas dimensionais...")
    # Gera dados dimensionais e obtém listas de IDs para joins
    dim_product_pd, product_ids = generate_product_data(NUM_PRODUCTS)
    dim_store_pd, store_ids = generate_store_data(NUM_STORES)

    print("Gerando tabelas fato...")
    # Gera dados fato utilizando os IDs dimensionais
    ft_inventory_pd = generate_inventory_data(NUM_INVENTORY_RECORDS, store_ids, product_ids)
    ft_sales_pd = generate_sales_data(NUM_SALES_RECORDS, store_ids, product_ids)
    
    print("Convertendo DataFrames pandas para DataFrames Spark...")
    # Converte DataFrames pandas para DataFrames Spark
    # A variável 'spark' está disponível automaticamente em notebooks Databricks
    dim_product_df = spark.createDataFrame(dim_product_pd)
    dim_store_df = spark.createDataFrame(dim_store_pd)
    ft_inventory_df = spark.createDataFrame(ft_inventory_pd)
    ft_sales_df = spark.createDataFrame(ft_sales_pd)

    print("Salvando DataFrames como tabelas no Databricks...")
    # Salva os DataFrames como tabelas no Databricks
    # Isso irá sobrescrever as tabelas caso já existam
    dim_product_df.write.mode("overwrite").saveAsTable("dbacademy.<seu_database>.dim_product")
    dim_store_df.write.mode("overwrite").saveAsTable("dbacademy.<seu_database>.dim_store")
    ft_inventory_df.write.mode("overwrite").saveAsTable("dbacademy.<seu_database>.ft_inventory")
    ft_sales_df.write.mode("overwrite").saveAsTable("dbacademy.<seu_database>.ft_sales")
    
    print("\n--- Processo Concluído ---")
    print(f"Tabela 'dim_product' criada com {dim_product_df.count()} registros.")
    print(f"Tabela 'dim_store' criada com {dim_store_df.count()} registros.")
    print(f"Tabela 'ft_inventory' criada com {ft_inventory_df.count()} registros.")
    print(f"Tabela 'ft_sales' criada com {ft_sales_df.count()} registros.")
    print("\nAgora você pode consultar essas tabelas usando SQL, ex: 'SELECT * FROM dim_product LIMIT 10'.")


   
## Exibir Tabelas

   
![](https://raw.githubusercontent.com/Databricks-BR/workshop_agents/refs/heads/main/demo-main/img/img/01_unity_catalog_delta.png)

O **Unity Catalog** oferece governança unificada e aberta para ativos de dados e IA, centralizando o gerenciamento de acesso, rastreando a linhagem de dados, suportando monitoramento e auditoria, e aprimorando a descoberta de dados — fornecendo às organizações uma fonte única e confiável de informações em diversas fontes e formatos. O **Delta Lake**, como camada de armazenamento fundamental, garante desempenho e confiabilidade por meio de transações ACID, controle de versão com time travel e suporte otimizado para dados em batch e streaming, possibilitando interoperabilidade em formatos abertos.

Juntos, o Unity Catalog e o Delta Lake oferecem uma solução integrada que combina governança robusta com armazenamento confiável e de alto desempenho para ambientes de dados modernos.

In [0]:
%sql
SELECT *
FROM dbacademy.<seu_database>.dim_product
LIMIT 10

In [0]:
%sql
SELECT *
FROM dbacademy.<seu_database>.dim_store
LIMIT 10

In [0]:
%sql
SELECT *
FROM dbacademy.<seu_database>.ft_inventory
LIMIT 10

In [0]:
%sql
SELECT *
FROM dbacademy.<seu_database>.ft_sales
LIMIT 10

   
## Gerar Metric Views

   
### Metric View de Inventário

In [0]:
%sql
    
CREATE OR REPLACE VIEW dbacademy.<seu_database>.mvw_inventory (
  `Date` COMMENT 'Data do registro de inventário (YYYY-MM-DD)',
  `Store ID` COMMENT 'Identificador da loja',
  `Store` COMMENT 'Nome ou identificador da loja',
  `Store Type` COMMENT 'Tipo ou categoria da loja',
  `Store Zip` COMMENT 'CEP da loja',
  `Store Lat Long` COMMENT 'Latitude e longitude da loja',
  `Retailer` COMMENT 'Nome do varejista ao qual a loja pertence',
  `Product` COMMENT 'Nome do produto',
  `Supplier` COMMENT 'Fornecedor do produto',
  `Inventory Quantity` COMMENT 'Soma total da quantidade de inventário disponível'
)
WITH METRICS
LANGUAGE YAML
COMMENT 'Metric View centralizada para análise de inventário disponível.'
AS $$

  version: 0.1

  # A tabela fato 'ft_inventory' é a fonte principal dos nossos dados numéricos.
  source: dbacademy.<seu_database>.ft_inventory

  # Unimos as tabelas dimensionais para enriquecer os dados de inventário
  # com atributos de loja e produto.
  joins:
  - name: dim_store
    source: dbacademy.<seu_database>.dim_store
    using:
    - store_id
  - name: dim_product
    source: dbacademy.<seu_database>.dim_product
    using:
    - product_id
  
  # Dimensões: São os atributos de negócio pelos quais os dados podem ser agrupados ou filtrados.
  # Vêm tanto da tabela fato (source) quanto das dimensões unidas.
  dimensions:
  - name: Date
    expr: source.date_key
  - name: Store ID
    expr: dim_store.store_id
  - name: Store
    expr: dim_store.store
  - name: Store Type
    expr: dim_store.store_type
  - name: Store Zip
    expr: dim_store.store_zip
  - name: Store Lat Long
    expr: dim_store.store_lat_long
  - name: Retailer
    expr: dim_store.retailer
  - name: Product
    expr: dim_product.product
  - name: Supplier
    expr: dim_product.supplier

  # Medidas: São os valores numéricos que são agregados (SUM, AVG, COUNT, etc.).
  measures:
  - name: Inventory Quantity
    expr: SUM(on_hand_quantity)

$$

In [0]:
%sql
SELECT
  `Store`,
  MEASURE(`Inventory Quantity`)
FROM dbacademy.<seu_database>.mvw_inventory
GROUP BY 1
ORDER BY 2 DESC

   
### Metric View de Vendas

In [0]:
%sql
    
CREATE OR REPLACE VIEW dbacademy.<seu_database>.mvw_sales (
  `Date` COMMENT 'Data da transação de venda (YYYY-MM-DD)',
  `Store` COMMENT 'Nome ou identificador da loja onde ocorreu a venda',
  `Store Type` COMMENT 'Tipo ou categoria da loja',
  `Store Zip` COMMENT 'CEP da loja',
  `Retailer` COMMENT 'Nome do varejista ao qual a loja pertence',
  `Product` COMMENT 'Nome do produto vendido',
  `Supplier` COMMENT 'Fornecedor do produto vendido',
  `Sales Quantity` COMMENT 'Soma total de unidades vendidas',
  `Sales Amount` COMMENT 'Valor total das vendas em moeda local',
  `Average Sales Ticket` COMMENT 'Métrica chave: Valor médio por unidade vendida (Sales Amount / Sales Quantity).'
)
WITH METRICS
LANGUAGE YAML
COMMENT 'Metric View centralizada para análise de desempenho de vendas.'
AS $$

  version: 0.1

  # A tabela fato 'ft_sales' é a fonte principal dos nossos dados numéricos.
  source: dbacademy.<seu_database>.ft_sales

  # Unimos as tabelas dimensionais para enriquecer os dados de vendas
  # com atributos de loja e produto.
  joins:
  - name: dim_store
    source: dbacademy.<seu_database>.dim_store
    using:
    - store_id
  - name: dim_product
    source: dbacademy.<seu_database>.dim_product
    using:
    - product_id
  
  # Dimensões: São os atributos de negócio pelos quais os dados podem ser agrupados ou filtrados.
  dimensions:
  - name: Date
    expr: source.date_key
  - name: Store
    expr: dim_store.store
  - name: Store Type
    expr: dim_store.store_type
  - name: Store Zip
    expr: dim_store.store_zip
  - name: Retailer
    expr: dim_store.retailer
  - name: Product
    expr: dim_product.product
  - name: Supplier
    expr: dim_product.supplier

  # Medidas: São os valores numéricos base que são agregados.
  measures:
  - name: Sales Quantity
    expr: SUM(sales_quantity)
  - name: Sales Amount
    expr: SUM(sales_amount)
    
  # --- Métricas Compostas ---
  # São recalculadas durante a agregação para serem precisas em qualquer nível de granularidade.
  # Usamos NULLIF para evitar erros de divisão por zero caso a quantidade de vendas seja 0.
  - name: Average Sales Ticket
    expr: SUM(sales_amount) / NULLIF(SUM(sales_quantity), 0)

$$

In [0]:
%sql
SELECT
  `Store`,
  MEASURE(`Sales Quantity`)
FROM dbacademy.<seu_database>.mvw_sales
GROUP BY 1
ORDER BY 2 DESC